## 自回归预训练

In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM,DataCollatorForLanguageModeling,Trainer,TrainingArguments,BloomForCausalLM
from datasets import load_dataset

In [2]:

ds = load_dataset("pleisto/wikipedia-cn-20230720-filtered")

In [3]:
ds
dataset = ds["train"]
dataset

Dataset({
    features: ['completion', 'source'],
    num_rows: 254547
})

In [4]:
dataset[0]

{'completion': '昭通机场（ZPZT）是位于中国云南昭通的民用机场，始建于1935年，1960年3月开通往返航班“昆明－昭通”，原来属军民合用机场。1986年机场停止使用。1991年11月扩建，于1994年2月恢复通航。是西南地区「文明机场」，通航城市昆明。 机场占地1957亩，飞行区等级为4C，有一条跑道，长2720米，宽48米，可供波音737及以下机型起降。机坪面积6600平方米，停机位2个，航站楼面积1900平方米。位于城东6公里处，民航路与金鹰大道交叉处。\n航点\n客服电话\n昭通机场客服电话：0870-2830004',
 'source': 'wikipedia.zh2307'}

## 数据集处理

In [14]:
tokenizer = AutoTokenizer.from_pretrained("Langboat/bloom-389m-zh",truncation_side="left",padding_side="left") # 关键设置：左截断     # 左对齐填充)

def process_func(examples):
    content = [e + tokenizer.eos_token for e in examples["completion"]]
    return tokenizer(content, max_length=384, truncation= True)

In [15]:
tokenized_dataset = dataset.map(process_func, batched= True, remove_columns= dataset.column_names)

Map: 100%|██████████| 254547/254547 [01:59<00:00, 2121.95 examples/s]


In [12]:
tokenized_dataset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 254547
})

In [13]:
tokenizer.eos_token , tokenizer.eos_token_id

('</s>', 2)

## 创建模型

In [16]:
model = AutoModelForCausalLM.from_pretrained("Langboat/bloom-389m-zh")

## 训练参数

In [18]:
args = TrainingArguments(
    output_dir= "./causal_llm",
    per_device_train_batch_size=4 ,
    gradient_accumulation_steps= 8,
    logging_steps= 10,
    num_train_epochs= 1
)

## 创建trainer

In [19]:
trainer = Trainer(args= args , model= model , train_dataset= tokenized_dataset , 
                  data_collator= DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm= False))

## 模型训练

In [21]:
trainer.train()

c:\Users\32721\anaconda3\envs\transformers\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


KeyboardInterrupt: 

## 模型推理

In [22]:
from transformers import pipeline

pipe = pipeline("text-generation",model= model , tokenizer= tokenizer)

Device set to use cpu


In [23]:
pipe("华南理工大学的软件工程专业", max_length= 128, do_sample = True)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


[{'generated_text': '华南理工大学的软件工程专业 电子工程学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院 应用电子学院'}]